### Build a complete GPT architecture by assembling all previously built components.

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Hyperparameters
VOCAB_SIZE = 5000

EMBED_DIM = 256

NUM_HEADS = 8

NUM_LAYERS = 6

BLOCK_SIZE = 128

DROPOUT = 0.1

In [10]:
import importlib

# 1. Use importlib to dynamically load the notebook 4 module
notebook04_module = importlib.import_module("ipynb.fs.full.04_TransformerBlock(GPT Decoder Block)")

# 2. Extract the class from the loaded module
TransformerBlock = notebook04_module.TransformerBlock


In [11]:
# Verify - Transformer block
block = TransformerBlock(
    EMBED_DIM,
    NUM_HEADS
)

x = torch.randn(
    2,
    128,
    EMBED_DIM
)

print(block(x).shape)

torch.Size([2, 128, 256])


## Build Mini GPT

In [12]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        block_size,
        num_heads,
        num_layers
    ):
        super().__init__()
        self.block_size = block_size
        
        # Token Embedding
        self.token_embedding_table = nn.Embedding(vocab_size, embed_dim)
        # Position Embedding
        self.position_embedding_table = nn.Embedding(block_size, embed_dim)
        
        # Transformer Blocks
        self.blocks = nn.Sequential(
            *[TransformerBlock(embed_dim, num_heads) for _ in range(num_layers)]
        )
        
        # LayerNorm & Language Modeling Head
        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        token_embeddings = self.token_embedding_table(idx)
        position_embeddings = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = token_embeddings + position_embeddings
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        #  FIX 1: Return statement added to the end of the forward function
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            
            # This calls the forward method, which will now successfully unpack!
            logits, _ = self(idx_cond)
            
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
            
        return idx #  FIX 2: Removed the duplicate, unreachable return statement from here

In [13]:
model = MiniGPT(

    vocab_size=VOCAB_SIZE,

    embed_dim=EMBED_DIM,

    block_size=BLOCK_SIZE,

    num_heads=NUM_HEADS,

    num_layers=NUM_LAYERS

)

In [14]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"{total_params:,}"
)

7,332,232


In [15]:
# dummy test
x = torch.randint(
    0,
    VOCAB_SIZE,
    (4,128)
)

logits, loss = model(
    x,
    x
)

print(logits.shape)

print(loss)

torch.Size([512, 5000])
tensor(8.6487, grad_fn=<NllLossBackward0>)
